# 🚀 Darpan Labs - Train Digital Twin Adapters on Google Colab

This notebook trains all 18 persona adapters using **Mistral-7B-Instruct-v0.2**.

**Why Mistral instead of Llama-2?**
- ✅ **No gated access** - Works immediately without approval
- ✅ **Better performance** - Often outperforms Llama-2-7b
- ✅ **Same size** - 7B parameters
- ✅ **Apache 2.0 license** - Fully open

**Requirements:**
- Google Colab Pro (for better GPU)
- HuggingFace token (free, no approval needed)
- Your codebase uploaded to Google Drive or GitHub

**Runtime Settings:**
- Runtime > Change runtime type > GPU (T4, A100, or V100)
- High-RAM if available

## Step 1: Check GPU Availability

In [ ]:
!nvidia-smi

## Step 2: Mount Google Drive (if using Drive for code storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3: Clone Repository or Upload Code

### Option A: From Google Drive

In [ ]:
# If you uploaded the zip to Google Drive
import os
os.chdir('/content')

# Extract the uploaded code
!unzip -q '/content/drive/MyDrive/darpan-whatif-simulator.zip' -d /content/
os.chdir('/content/darpan-whatif-simulator')
!pwd

### Option B: From GitHub

In [ ]:
# Clone from your GitHub repository
!git clone https://github.com/aniketm-dl/mvp_v1.0.git
import os
os.chdir('/content/mvp_v1.0')
!pwd

## Step 4: Install Dependencies

In [ ]:
!pip install -q torch transformers>=4.42.0 peft>=0.10.0 accelerate>=0.30.0 datasets>=2.20.0 bitsandbytes sentencepiece protobuf

## Step 5: Authenticate with HuggingFace

Get your token from: https://huggingface.co/settings/tokens

**Note:** Mistral doesn't require special approval - any HuggingFace account works!

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Step 6: Verify Mistral-7B Access

In [ ]:
from transformers import AutoTokenizer

try:
    tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")
    print("✅ Mistral-7B access verified!")
    print(f"✅ Tokenizer vocabulary size: {len(tokenizer)}")
    print("\n🎉 No approval needed - ready to train!")
except Exception as e:
    print(f"❌ Error: {e}")
    print("\nMake sure you used a valid HuggingFace token")

## Step 7: Check Training Data

In [ ]:
import os
import json
from pathlib import Path

# List training data files
sft_dir = Path("DATA/sft")
if sft_dir.exists():
    files = list(sft_dir.glob("*.jsonl"))
    print(f"✅ Found {len(files)} training data files:")
    for f in sorted(files)[:5]:
        size_kb = f.stat().st_size / 1024
        print(f"   • {f.name} ({size_kb:.1f}KB)")
    if len(files) > 5:
        print(f"   ... and {len(files) - 5} more")
else:
    print("❌ DATA/sft directory not found!")

## Step 8: Train a Single Twin (Test Run)

Test with one persona first to ensure everything works.

In [ ]:
# Test with bargain_hunter first
!python scripts/train_llm_persona_sft.py \
  --twin_id bargain_hunter \
  --base_model mistralai/Mistral-7B-Instruct-v0.2 \
  --epochs 1 \
  --max_length 512 \
  --lr 2e-4

## Step 9: Verify Test Training

In [ ]:
adapter_file = Path("artifacts/llm_adapters/bargain_hunter/adapter_model.safetensors")
if adapter_file.exists():
    size_mb = adapter_file.stat().st_size / (1024 * 1024)
    print(f"✅ Test training successful! Adapter: {size_mb:.1f}MB")
    print(f"\n🎉 Mistral training works! Ready for full training.")
else:
    print("❌ Test training failed - adapter not created")

## Step 10: Train ALL 18 Personas

⚠️ **This will take 2-5 hours depending on GPU**

Colab Pro GPU estimates:
- T4: ~10-15 min per twin = 3-4 hours total
- V100/A100: ~5-8 min per twin = 1.5-2.5 hours total

**Mistral typically trains slightly faster than Llama-2!**

In [ ]:
import time
start_time = time.time()

!python scripts/train_all_adapters.py

elapsed = (time.time() - start_time) / 60
print(f"\n⏱️ Total training time: {elapsed:.1f} minutes ({elapsed/60:.1f} hours)")

## Step 11: Verify All Adapters

In [ ]:
import json
from pathlib import Path

# Load personas
personas = json.loads(Path("DATA/personas.json").read_text())["personas"]

print("📊 Adapter Training Summary:\n")
print(f"{'Twin ID':<25} {'Status':<10} {'Size (MB)':<12}")
print("=" * 50)

total_size = 0
success_count = 0

for persona in personas:
    twin_id = persona["id"]
    adapter_file = Path(f"artifacts/llm_adapters/{twin_id}/adapter_model.safetensors")
    
    if adapter_file.exists():
        size_mb = adapter_file.stat().st_size / (1024 * 1024)
        total_size += size_mb
        success_count += 1
        print(f"{twin_id:<25} {'✅ Success':<10} {size_mb:>10.1f}")
    else:
        print(f"{twin_id:<25} {'❌ Missing':<10} {'-':>10}")

print("=" * 50)
print(f"\n✅ Successfully trained: {success_count}/{len(personas)}")
print(f"💾 Total adapter size: {total_size:.1f}MB")

if success_count == len(personas):
    print(f"\n🎉 All {len(personas)} Mistral-powered twins ready!")

## Step 12: Package Adapters for Download

In [ ]:
# Create a zip file of all trained adapters
!cd artifacts && zip -r llm_adapters_mistral.zip llm_adapters/

# Check size
import os
zip_size = os.path.getsize("artifacts/llm_adapters_mistral.zip") / (1024 * 1024)
print(f"\n📦 Package created: llm_adapters_mistral.zip ({zip_size:.1f}MB)")

## Step 13: Download to Google Drive

In [ ]:
# Copy to Google Drive for easy download
!cp artifacts/llm_adapters_mistral.zip /content/drive/MyDrive/

print("✅ Adapters saved to Google Drive: MyDrive/llm_adapters_mistral.zip")
print("\nYou can now:")
print("1. Download from Google Drive to your local machine")
print("2. Extract to: artifacts/llm_adapters/")
print("3. Run: python interact_cli.py")
print("\n🎉 Mistral-powered digital twins ready to use!")

## Step 14: Test an Adapter (Optional)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

# Load base model
base_model = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Load adapter
twin_id = "bargain_hunter"
model = PeftModel.from_pretrained(model, f"artifacts/llm_adapters/{twin_id}")

# Test prompt
prompt = "[INST] You are The Bargain Hunter. Be concise. 1-2 sentences. Should I buy this $50 product or wait for a sale? [/INST]"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    temperature=0.3,
    top_p=0.9,
    do_sample=True
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n" + "="*60)
print("BARGAIN HUNTER (Mistral-powered):")
print(response.split("[/INST]")[-1].strip())
print("="*60)

## 🎉 Training Complete!

### What you have now:
- ✅ 18 Mistral-7B fine-tuned adapters
- ✅ Better performance than Llama-2
- ✅ No licensing restrictions
- ✅ Ready for deployment

### Next steps:
1. Download `llm_adapters_mistral.zip` from Google Drive
2. Extract locally: `python scripts/colab/download_adapters.py --zip ~/Downloads/llm_adapters_mistral.zip --backup`
3. Test: `python interact_cli.py`
4. Deploy: `uvicorn src.api.service:app --reload`

### Advantages of Mistral over Llama-2:
- 🚀 **Faster training** - Slightly more efficient
- 💬 **Better instruction following** - More natural responses
- 🔓 **No gates** - Instant access, no approval wait
- 📜 **Apache 2.0** - Fully permissive license